# An Attribution-Controlled Study of Feature Priors in LLM-Guided Grammatical Evolution



In [1]:
import sys
from pathlib import Path

# --- locate the package, wherever the kernel's working directory happens to be ---
_SKIP = {".git", ".ipynb_checkpoints", "__pycache__", "node_modules", ".venv",
         "venv", "env", ".idea", ".vscode", "data", "results", "figures"}


def _search_downwards(base, package, max_depth=3):
    frontier, depth = [Path(base)], 0
    while frontier and depth <= max_depth:
        nxt = []
        for folder in frontier:
            if (folder / package / "__init__.py").exists():
                return folder
            try:
                children = [c for c in folder.iterdir()
                            if c.is_dir() and c.name not in _SKIP
                            and not c.name.startswith(".")]
            except (PermissionError, OSError):
                continue
            nxt.extend(children)
        frontier, depth = nxt, depth + 1
    return None


def add_repository_root(package="geprior"):
    """Put the repository root on sys.path. Jupyter and VS Code disagree about a
    notebook's working directory, so both directions are searched."""
    roots = [Path.cwd().resolve()]
    notebook = globals().get("__vsc_ipynb_file__")     # set by VS Code only
    if notebook:
        roots.append(Path(notebook).resolve().parent)
    for root in roots:
        for base in [root, *root.parents]:
            if (base / package / "__init__.py").exists():
                if str(base) not in sys.path:
                    sys.path.insert(0, str(base))
                return base
    for root in roots:
        found = _search_downwards(root, package)
        if found is not None:
            if str(found) not in sys.path:
                sys.path.insert(0, str(found))
            return found
    raise ModuleNotFoundError(
        f"could not locate the {package!r} package. Unpack the full project so that "
        f"the {package}/ folder sits alongside this notebook, or add its parent with "
        f"sys.path.insert(0, r'C:\\path\\to\\ge_prior_attribution').")


ROOT = add_repository_root()

# --- install any missing dependency into the interpreter actually running here ---
REQUIRED_PACKAGES = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
                     "sklearn": "scikit-learn", "deap": "deap",
                     "matplotlib": "matplotlib"}


def ensure_dependencies():
    import importlib, importlib.util, subprocess
    missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
               if importlib.util.find_spec(module) is None]
    if not missing:
        return
    print("kernel interpreter:", sys.executable)
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()


ensure_dependencies()

import numpy as np
import pandas as pd
from geprior import config, datasets, priors, grammar, engine
from geprior import experiments, baselines, statistics as st, figures as fg
pd.set_option("display.width", 160)

# --- quick-check toggle -------------------------------------------------------
# SMOKE = True runs a tiny version end to end and writes to results/_smoke so it
# does not touch the full-campaign checkpoints. Set to False for the real 30-seed run.
SMOKE = False
if SMOKE:
    import dataclasses
    config.GE = dataclasses.replace(config.GE, population_size=40, n_generations=6,
                                    hall_of_fame_size=8)
    config.N_SEEDS = 3
    config.ABLATION_SEEDS = 2
    config.ENSEMBLE_SIZE = 5
    config.RESULTS_DIR = config.RESULTS_DIR / "_smoke"
    config.FIGURES_DIR = config.FIGURES_DIR / "_smoke"
    config.RESULTS_DIR.mkdir(exist_ok=True)
    config.FIGURES_DIR.mkdir(exist_ok=True)

KEYS = ["wbcd", "pima", "cleveland", "heart_failure"]
ARMS = list(config.SELECTION_ARMS)
CONSTRUCTION = list(config.CONSTRUCTION_ARMS)
SEEDS = range(1, config.N_SEEDS + 1)
print("repository root:", ROOT)
print(config.summary())

repository root: C:\Users\awwal\OneDrive\Desktop\ge_prior_attribution\ge_prior_attribution
seeds=30 pop=200 gens=40 fitness=auc grammar=weighted threshold=calibrated ensemble=9 backend=cached


## 1. Datasets

Four public clinical datasets spanning a range of difficulty, so an attribution effect that only appears away from a performance ceiling is not masked by a saturated benchmark. `K` is the fixed selection budget shared by every selection-prior arm.

In [2]:
datasets.describe()

,key,dataset,samples,features,budget,positive_rate
0,wbcd,WBCD,569,30,8,0.373
1,pima,Pima,768,8,4,0.349
2,cleveland,Cleveland,303,13,6,0.545
3,heart_failure,Heart Failure,299,11,5,0.321


## 2. The knowledge prior

The prior is an LLM output reasoning over feature names alone, with no access to labels or data statistics. `LLM_BACKEND = "cached"` replays a provenance-documented response; setting it to `"api"` with `ANTHROPIC_API_KEY` draws a fresh response per seed.

In [3]:
print("backend:", priors.LLM.backend, "| live:", priors.LLM.is_live)
print("provenance:", priors.CACHED_LLM_PRIOR["provenance"])
for key in KEYS:
    split = datasets.prepare_split(key, seed=1)
    idx = priors.selection_indices("llm", split.X_train, split.y_train,
                                   split.features, split.budget, 1, key)
    print(f"  {key:14s}", [split.features[i] for i in idx])

backend: cached | live: False
provenance: {'model': 'Claude (Anthropic)', 'method': 'clinical reasoning over feature names; no access to labels or statistics', 'temperature': 0.0}
  wbcd           [np.str_('mean concavity'), np.str_('mean concave points'), np.str_('worst radius'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst compactness'), np.str_('worst concavity'), np.str_('worst concave points')]
  pima           ['Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age']
  cleveland      ['cp', 'thalach', 'exang', 'oldpeak', 'ca', 'thal']
  heart_failure  ['age', 'anaemia', 'creatinine_phosphokinase', 'ejection_fraction', 'serum_creatinine']


## 3. Grammar

A prior is injected purely by restricting the terminal rule `<v>`; every other production is identical across arms, so the only degree of freedom that differs between conditions is which feature indices the grammar can reach.

In [4]:
split = datasets.prepare_split("pima", seed=1)
idx = priors.selection_indices("llm", split.X_train, split.y_train,
                               split.features, split.budget, 1, "pima")
print(grammar.weighted_grammar_text(idx))

<e> ::= <t> | add(<e>,<t>) | sub(<e>,<t>)
<t> ::= mul(<w>,<v>) | mul(<w>,<nl>)
<nl> ::= mul(<v>,<v>) | pdiv(<v>,<v>) | psqrt(<v>) | plog(<v>)
<v> ::= x[:,1] | x[:,5] | x[:,6] | x[:,7]
<w> ::= 0.1 | 0.25 | 0.5 | 0.75 | 1.0 | 1.5 | 2.0 | 3.0 | 5.0



## 4. Baselines

Five standard classifiers on the full unrestricted feature set, fitted on the identical per-seed splits as every GE condition, so the tables are directly comparable rather than computed on a different resampling.

In [5]:
base = baselines.run_baselines(seeds=SEEDS)
base.groupby(["model", "dataset"])["roc_auc"].median().unstack("dataset")[KEYS].round(3)

  baselines: wbcd complete (31s)


c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

  baselines: pima complete (147s)


c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(


  baselines: cleveland complete (192s)


c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\awwal\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

  baselines: heart_failure complete (217s)


dataset,wbcd,pima,cleveland,heart_failure
model,,,,
Gradient Boosting,0.992,0.829,0.875,0.741
Logistic Regression,0.997,0.838,0.907,0.769
MLP,0.995,0.771,0.884,0.708
Random Forest,0.991,0.828,0.903,0.783
SVM (RBF),0.997,0.826,0.904,0.778


### Black-box feature-engineering comparator

A lightweight, faithful-in-spirit reimplementation of the black-box LLM feature-engineering paradigm (CAAFE; Hollmann et al., 2023): LLM-proposed features added greedily under a cross-validated acceptance gate, feeding a logistic regression.

In [6]:
blackbox = baselines.run_blackbox_baseline(seeds=SEEDS)
blackbox.groupby("dataset")["roc_auc"].median().reindex(KEYS).round(3)

  black-box FE: wbcd complete (3s)
  black-box FE: pima complete (5s)
  black-box FE: cleveland complete (8s)
  black-box FE: heart_failure complete (10s)


dataset
wbcd             0.997
pima             0.838
cleveland        0.906
heart_failure    0.770
Name: roc_auc, dtype: float64

## 5. Selection study

Noise, data and knowledge as sources of a feature prior, at matched budget. Pass `time_budget=<seconds>` to run in bounded slices; re-run the cell to continue from the checkpoint.

In [7]:
selection = experiments.run_selection_study(seeds=SEEDS, n_restarts=1)
st.median_table(selection, ARMS, dataset_keys=KEYS).round(3)

  selection: wbcd complete (290s)
  selection: pima complete (684s)
  selection: cleveland complete (1143s)
  selection: heart_failure complete (1449s)


arm,none,random,mi,corr,llm
dataset,,,,,
wbcd,0.991,0.987,0.987,0.987,0.988
pima,0.828,0.775,0.828,0.827,0.835
cleveland,0.865,0.830,0.868,0.859,0.877
heart_failure,0.752,0.669,0.771,0.768,0.785


## 6. Construction study

GE runs unrestricted over the raw feature set augmented with five engineered features. Constructions are computed on raw-scale values and the augmented matrix is re-standardised on training rows only.

In [8]:
construction = experiments.run_construction_study(seeds=SEEDS)
st.median_table(construction, CONSTRUCTION, dataset_keys=KEYS).round(3)

  construction: wbcd complete (174s)
  construction: pima complete (407s)
  construction: cleveland complete (720s)
  construction: heart_failure complete (937s)


arm,raw,random,llm
dataset,,,
wbcd,0.991,0.991,0.990
pima,0.828,0.826,0.830
cleveland,0.865,0.854,0.858
heart_failure,0.752,0.747,0.747


## 7. Selection-prior figure

In [9]:
fg.plot_arm_medians(selection, ARMS, KEYS, "selection_by_dataset",
                    title="Selection-prior arms by dataset (median over seeds)").round(3)

arm,none,random,mi,corr,llm
dataset,,,,,
wbcd,0.991,0.987,0.987,0.987,0.988
pima,0.828,0.775,0.828,0.827,0.835
cleveland,0.865,0.830,0.868,0.859,0.877
heart_failure,0.752,0.669,0.771,0.768,0.785


## 8. Statistical protocol

Per-dataset Friedman tests over the seed-paired resamples carry the inference. With only four datasets a cross-dataset critical-difference diagram would be under-powered, so the mean rank is reported descriptively and no CD diagram is produced.

In [10]:
friedman, ranks = st.per_dataset_friedman(selection, ARMS, dataset_keys=KEYS)
display(friedman.round(4))
print("mean rank across datasets (descriptive only):")
print(ranks.mean(axis=0).round(2).to_string())

,dataset,n_seeds,friedman_chi2,p,none,random,mi,corr,llm
0,wbcd,30,13.33,0.0098,2.12,3.45,3.28,3.02,3.13
1,pima,30,52.08,0.0000,3.07,4.75,2.48,2.60,2.10
2,cleveland,30,23.57,0.0001,2.90,3.97,2.62,3.35,2.17
3,heart_failure,30,32.18,0.0000,3.37,4.23,2.62,2.63,2.15


mean rank across datasets (descriptive only):
arm
none      2.86
random    4.10
mi        2.75
corr      2.90
llm       2.39


In [11]:
fg.plot_rank_heatmap(ranks.reindex(KEYS))

### Pre-registered contrasts

C1 knowledge versus noise, C2 knowledge versus data, C3 the construction analogue of C1. Fixed before any run, Holm-corrected within each dataset, Cliff's delta reported with every pairwise claim.

In [12]:
contrasts = st.preregistered_contrasts(selection, construction, dataset_keys=KEYS)
contrasts.round(4)

,dataset,contrast_id,study,contrast,median_diff,cliffs_delta,magnitude,p_raw,p_holm,significant,n
0,wbcd,C1,selection,llm vs random,0.0016,0.0678,negligible,0.2801,0.8403,False,30
1,wbcd,C2,selection,llm vs mi,0.0006,0.0333,negligible,0.7000,1.0000,False,30
2,wbcd,C3,construction,llm vs random,0.0009,-0.0067,negligible,0.6583,1.0000,False,30
3,pima,C1,selection,llm vs random,0.0531,0.7856,large,0.0000,0.0000,True,30
4,pima,C2,selection,llm vs mi,0.0021,0.1678,small,0.2494,0.4343,False,30
5,pima,C3,construction,llm vs random,0.0020,0.1511,small,0.2172,0.4343,False,30
6,cleveland,C1,selection,llm vs random,0.0535,0.5878,large,0.0000,0.0000,True,30
7,cleveland,C2,selection,llm vs mi,0.0061,0.1944,small,0.0306,0.0612,False,30
8,cleveland,C3,construction,llm vs random,0.0006,0.1056,negligible,0.6554,0.6554,False,30
9,heart_failure,C1,selection,llm vs random,0.0793,0.6467,large,0.0000,0.0000,True,30


## 9. Structural 

Diagnostics pooled across datasets: effective length is used codons, structural diversity is the proportion of distinct derivation structures in the final population.

In [13]:
display(st.structural_diagnostics(selection, ARMS))
fg.plot_length_vs_generalisation(selection, ARMS)

,effective_length,structural_diversity,phenotype_coverage,invalids,depth,generalisation_gap,test_roc_auc
arm,,,,,,,
none,32.0,0.893,0.462,0.0,10.0,0.012,0.841
random,33.0,0.885,0.800,0.0,10.0,0.024,0.797
mi,33.0,0.885,0.800,0.0,10.0,0.013,0.845
corr,33.5,0.880,0.833,0.0,10.5,0.013,0.844
llm,35.0,0.895,0.854,0.0,11.0,0.011,0.853


## 10. What interpretability costs

A single symbolic rule against a bagged black-box comparator conflates two things: the cost of the symbolic representation and the cost of comparing one model to an ensemble. Reporting a protocol-matched GE ensemble alongside the single rule separates them.

In [14]:
ensemble = experiments.run_selection_study(
    seeds=SEEDS, arms=["llm"], n_restarts=5,
    checkpoint=config.RESULTS_DIR / "ensemble.csv")
comparison = pd.DataFrame({
    "GE rule (llm)": selection[selection.arm == "llm"].groupby("dataset")["test_roc_auc"].median(),
    "GE ensemble (llm)": ensemble.groupby("dataset")["ensemble_roc_auc"].median(),
    "Black-box FE": blackbox.groupby("dataset")["roc_auc"].median(),
    "Logistic Regression": base[base.model == "Logistic Regression"].groupby("dataset")["roc_auc"].median(),
}).reindex(KEYS)
fg.plot_interpretability_cost(comparison)
comparison.round(3)

  selection: wbcd complete (296s)
  selection: pima complete (659s)
  selection: cleveland complete (1031s)
  selection: heart_failure complete (1377s)


,GE rule (llm),GE ensemble (llm),Black-box FE,Logistic Regression
dataset,,,,
wbcd,0.988,0.989,0.997,0.997
pima,0.835,0.840,0.838,0.838
cleveland,0.877,0.890,0.906,0.907
heart_failure,0.785,0.776,0.770,0.769


## 11. Configuration ablation

Each variant adds exactly one configuration choice to the one above it, so the effect is attributed to a specific choice rather than to the pipeline as a whole.

In [15]:
ablation = experiments.run_ablation(seeds=range(1, config.ABLATION_SEEDS + 1))
display(ablation.groupby(["dataset", "variant"])["test_roc_auc"].median()
        .unstack("variant").reindex(KEYS).round(3))
display(ablation.groupby(["dataset", "variant"])["test_f1"].median()
        .unstack("variant").reindex(KEYS).round(3))
fg.plot_configuration_ablation(ablation)

  ablation: wbcd complete (297s)
  ablation: pima complete (709s)
  ablation: cleveland complete (1201s)
  ablation: heart_failure complete (1583s)


variant,+ AUC objective,+ threshold calibration,+ validation selection,+ weighted grammar,Base
dataset,,,,,
wbcd,0.985,0.988,0.988,0.987,0.982
pima,0.826,0.835,0.835,0.834,0.821
cleveland,0.874,0.877,0.877,0.879,0.828
heart_failure,0.760,0.785,0.785,0.785,0.740


variant,+ AUC objective,+ threshold calibration,+ validation selection,+ weighted grammar,Base
dataset,,,,,
wbcd,0.908,0.922,0.913,0.926,0.911
pima,0.633,0.676,0.660,0.658,0.624
cleveland,0.812,0.820,0.795,0.798,0.804
heart_failure,0.573,0.611,0.545,0.544,0.588


variant,Base,+ AUC objective,+ weighted grammar,+ validation selection,+ threshold calibration
dataset,,,,,
cleveland,0.828293,0.873902,0.878902,0.877439,0.877439
heart_failure,0.739542,0.759751,0.785472,0.785472,0.785472
pima,0.820741,0.826255,0.834115,0.835391,0.835391
wbcd,0.981893,0.984886,0.986711,0.988026,0.988026


## 12. Representative interpretable rules

The run whose test ROC-AUC is nearest that arm's **median**, not its best. Selecting the best-scoring or highest-train-accuracy run would showcase an optimistic outlier.

In [16]:
def representative_rule(df, key, arm="llm"):
    sub = df[(df.dataset == key) & (df.arm == arm)].dropna(subset=["test_roc_auc"])
    target = sub["test_roc_auc"].median()
    row = sub.iloc[(sub["test_roc_auc"] - target).abs().argmin()]
    split = datasets.prepare_split(key, int(row["seed"]))
    return dict(dataset=key, seed=int(row["seed"]),
                roc_auc=round(row["test_roc_auc"], 3), f1=round(row["test_f1"], 3),
                effective_length=int(row["effective_length"]),
                rule=engine.readable_rule(row["phenotype"], split.features))


for key in KEYS:
    r = representative_rule(selection, key)
    print(f"\n{r['dataset']}  AUC={r['roc_auc']}  F1={r['f1']}  len={r['effective_length']}")
    print("  ", r["rule"][:220])


wbcd  AUC=0.988  F1=0.91  len=39
   add(add(add(add(add(add(add(mul(1.5,mean concave points),mul(1.0,mul(worst area,mean concavity))),mul(2.0,worst concave points)),mul(1.5,worst perimeter)),mul(3.0,worst area)),mul(1.5,plog(worst radius))),mul(1.5,mul(wor

pima  AUC=0.836  F1=0.591  len=55
   add(add(add(add(add(add(add(add(add(add(add(mul(2.0,DiabetesPedigreeFunction),mul(0.25,mul(DiabetesPedigreeFunction,BMI))),mul(1.5,Age)),mul(1.0,Glucose)),mul(1.0,BMI)),mul(1.5,Age)),mul(1.0,Glucose)),mul(3.0,BMI)),mul(0

cleveland  AUC=0.879  F1=0.821  len=39
   sub(add(sub(add(sub(add(sub(sub(mul(0.5,ca),mul(1.0,thal)),mul(1.5,plog(cp))),mul(1.0,cp)),mul(2.0,ca)),mul(0.5,thalach)),mul(0.1,psqrt(thal))),mul(0.25,thalach)),mul(3.0,psqrt(thal)))

heart_failure  AUC=0.783  F1=0.598  len=43
   add(add(add(sub(add(add(add(add(add(mul(3.0,serum_creatinine),mul(3.0,plog(ejection_fraction))),mul(0.5,serum_creatinine)),mul(1.5,psqrt(ejection_fraction))),mul(3.0,plog(age))),mul(0.5,creatinine_phosphokina

## 13. Explainability check on WBCD

A surrogate explainer. The SHAP values below are computed on a **random forest**, not on the evolved rule: they are an independent check on which features a conventional black-box model considers important, and must not be described as an explanation of the evolved model itself.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

rep = representative_rule(selection, "wbcd")
split = datasets.prepare_split("wbcd", rep["seed"])
row = selection[(selection.dataset == "wbcd") & (selection.arm == "llm")
                & (selection.seed == rep["seed"])].iloc[0]
scores = engine.phenotype_score(row["phenotype"], split.X_test)
fg.plot_confusion(split.y_test, (scores > row["threshold"]).astype(int),
                  "WBCD evolved rule, test partition", "wbcd_confusion")
forest = RandomForestClassifier(n_estimators=400,
                                random_state=config.GLOBAL_SEED).fit(split.X_train, split.y_train)
order = np.argsort(forest.feature_importances_)[::-1][:6]
print("random-forest surrogate, top-6 features:", [split.features[i] for i in order])
print("features referenced by the evolved rule:",
      [split.features[i] for i in engine.used_feature_indices(row["phenotype"], split.n_features)])

random-forest surrogate, top-6 features: [np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst concave points'), np.str_('mean concave points'), np.str_('mean concavity'), np.str_('worst radius')]
features referenced by the evolved rule: [np.str_('mean concavity'), np.str_('mean concave points'), np.str_('worst radius'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst compactness'), np.str_('worst concave points')]


## 14. Consolidated summary

In [18]:
print("=== baselines ===")
print(base.groupby(["model", "dataset"])["roc_auc"].median().unstack("dataset")[KEYS].round(3).to_string())
print("\n=== selection arms (median test ROC-AUC) ===")
print(st.median_table(selection, ARMS, dataset_keys=KEYS).round(3).to_string())
print("\n=== construction arms (median test ROC-AUC) ===")
print(st.median_table(construction, CONSTRUCTION, dataset_keys=KEYS).round(3).to_string())
print("\n=== per-dataset Friedman ===")
print(friedman.round(4).to_string(index=False))
print("\n=== pre-registered contrasts ===")
print(contrasts.round(4).to_string(index=False))
print("\n=== structural diagnostics ===")
print(st.structural_diagnostics(selection, ARMS).to_string())
print("\n=== interpretability cost ===")
print(comparison.round(3).to_string())

=== baselines ===
dataset               wbcd   pima  cleveland  heart_failure
model                                                      
Gradient Boosting    0.992  0.829      0.875          0.741
Logistic Regression  0.997  0.838      0.907          0.769
MLP                  0.995  0.771      0.884          0.708
Random Forest        0.991  0.828      0.903          0.783
SVM (RBF)            0.997  0.826      0.904          0.778

=== selection arms (median test ROC-AUC) ===
arm             none  random     mi   corr    llm
dataset                                          
wbcd           0.991   0.987  0.987  0.987  0.988
pima           0.828   0.775  0.828  0.827  0.835
cleveland      0.865   0.830  0.868  0.859  0.877
heart_failure  0.752   0.669  0.771  0.768  0.785

=== construction arms (median test ROC-AUC) ===
arm              raw  random    llm
dataset                            
wbcd           0.991   0.991  0.990
pima           0.828   0.826  0.830
cleveland      0.865   